In [1]:
from pennylane.fermi import FermiSentence, from_string
import pennylane as qml
from pennylane import numpy as np
import time


qml.about()


Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_catalyst, pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)


In [2]:
import jax
print('✅ JAX version:', jax.devices())
import jax
print('✅ JAX version:', jax.__version__)
print('✅ Devices:', jax.devices())
if any(d.platform == 'gpu' for d in jax.devices()):
    print('🎉 GPU is working!')
else:
    print('⚠️ No GPU detected')

✅ JAX version: [CudaDevice(id=0)]
✅ JAX version: 0.6.2
✅ Devices: [CudaDevice(id=0)]
🎉 GPU is working!


In [3]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=6 n=3.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_3 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_3.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_3.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_3.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_3.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=3, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

✅ 矩阵调用成功！
矩阵格式：csr
矩阵形状：(309, 309)
非零元素个数：7765
最小特征值: [-8.48179737  4.02035168 30.88551057]
-8.481797373159006


In [4]:
import pennylane as qml
import jax
import jax.numpy as jnp
import numpy as np  # 用于预处理，非计算图部分
import time
import optax  # JAX 的标准优化库 (需要 pip install optax)
from catalyst import qjit, grad

# =================== 0. 辅助函数 (保持不变) ===================
def get_Hami(H):
    # 安全地将 H 转换为无梯度的 NumPy 数组
    if hasattr(H, 'toarray'):
        H = H.toarray()
    if hasattr(H, 'detach'):
        H = H.detach().cpu().numpy()
    elif hasattr(H, 'numpy'):
        H = H.numpy()
    else:
        H = np.asarray(H)

    H_dense = np.array(H, copy=True)
    d = H_dense.shape[0]
    Nq = int(np.ceil(np.log2(d)))
    l = 2 ** Nq

    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    np.fill_diagonal(Hami, 1000)
    Hami[:d, :d] = H_dense

    return Hami, Nq

H_sy,Nq = get_Hami(H)

from scipy.linalg import eigh

# 假设 H_sy 是一个 n x n 的实对称矩阵
eigenvals, eigenvecs = eigh(H_sy)

# 输出所有特征值（已按升序排列）
print("所有特征值（对角化结果）:")
print(eigenvals)

min =  eigh( H_sy,eigvals_only=True)[0]
print(min)
print("最小特征值:", min_eigval[0])

所有特征值（对角化结果）:
[  -8.48179737    4.02035168   30.88551057   45.23450166   66.45691138
   81.09186071   92.06933989   95.07905141  101.46808339  118.71061877
  124.08500927  128.60170062  139.15686327  144.80083995  149.75603552
  155.46116035  162.45195514  170.34440405  182.0271591   186.42175424
  193.34129407  194.96783193  199.96282976  201.51040452  202.52457944
  213.42209868  217.28219878  217.884753    220.62672944  224.55585091
  230.95176419  236.27867601  236.69061898  238.27751675  243.45636784
  244.93318541  250.76813217  251.55390408  255.69392701  259.00386696
  264.39373101  264.99379728  270.03277259  270.03277259  270.03277259
  272.98727615  279.99800375  284.05146881  284.80838613  288.09385023
  292.57198678  294.72244699  296.54841829  298.39467918  300.27589886
  301.30448872  303.78686916  303.78686916  303.78686916  306.15285145
  315.19721874  316.82303495  317.58580508  320.7652136   323.50556111
  328.90531568  330.9099755   331.56935543  333.98945611  335.9

In [5]:
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np

H_sy,Nq = get_Hami(H)
H_sy = coo_matrix(H_sy)
# ---------- 1. 生成 Gray 码 ----------
def gray_code(n: int) -> list[str]:
    """返回 n-bit Gray 码列表，顺序对应 0..2^n-1"""
    if n == 1:
        return ["0", "1"]
    lower = gray_code(n - 1)
    return ["0" + x for x in lower] + ["1" + x for x in reversed(lower)]

gray_basis   = gray_code(Nq)                        # len == 2**Nq
gray2natural = np.array([int(g, 2) for g in gray_basis], dtype=np.int64)

# ---------- 2. 构造 Gray-ordered 哈密顿量 ----------

rows, cols = H_sy.nonzero()          # 默认是 int32/uint32
data       = H_sy.data

# 关键修复：先把索引数组变成 int64，再做高级索引
rows = rows.astype(np.int64, copy=False)
cols = cols.astype(np.int64, copy=False)

new_rows = gray2natural[rows]
new_cols = gray2natural[cols]

# 现在三者长度一致，且索引不会溢出
H_gray = coo_matrix((data, (new_rows, new_cols)),
                    shape=(2**Nq, 2**Nq)).tocsr()

H_gray = H_gray.toarray()


H_hermitian = qml.Hermitian(H_gray, wires=range(Nq))
print(Nq)




9


In [6]:
import pennylane as qml
import numpy as np

def matrix_to_pauli_sentence(A, n_wires, tol=1e-10):
    A = np.asarray(A)
    # 若你确定 A 已经是 Hermitian，可删掉下一行
    A = 0.5 * (A + A.conj().T)

    wire_order = list(range(n_wires))
    ps = qml.pauli_decompose(A, pauli=True, wire_order=wire_order)
    return ps, wire_order

In [7]:
def pauliword_to_string(pw, wire_order):
    # pw 是 PauliWord，行为类似 dict：wire -> 'X'/'Y'/'Z'，缺省视为 'I'
    return "".join(pw[w] if w in pw else "I" for w in wire_order)

def build_initial_pool_from_pauli_sentence(ps, wire_order, drop_identity=True):
    pool = []
    n = len(wire_order)
    for pw, coeff in ps.items():
        pstr = pauliword_to_string(pw, wire_order)
        if drop_identity and pstr == "I"*n:
            continue
        pool.append(pstr )
    return pool


In [8]:
A = H_gray
n = 9

ps, wire_order = matrix_to_pauli_sentence(A, n_wires=n)

H = qml.pauli_decompose(A, wire_order=wire_order)

pool = build_initial_pool_from_pauli_sentence(ps, wire_order)



In [9]:
print(len(pool))
print(pool[0])

104707
IIIIIIIIX


In [10]:
import random
sampled_pool = random.sample(pool, 10000)
'''
operator_pool = []
for i in range(len(sampled_pool)):
    operator_pool.append(qml.PauliRot(0.0, sampled_pool[i], wires=range(n)))
'''

'\noperator_pool = []\nfor i in range(len(sampled_pool)):\n    operator_pool.append(qml.PauliRot(0.0, sampled_pool[i], wires=range(n)))\n'

In [11]:

def circuit_1(params, pool):
    qml.BasisState(np.zeros(n), wires=range(n))
    for i in range(n):
        qml.Hadamard(wires=i)
    for i in range(len(pool)):

        qml.PauliRot(params[i], pool[i], wires=range(n))

    return qml.expval(H_hermitian)


In [12]:
dev = qml.device("lightning.gpu", wires=n)
cost_fn = qml.QNode(circuit_1, dev, interface="jax")
circuit_gradient = jax.grad(cost_fn, argnums=0)

params = [0.0] * len(sampled_pool)

grads = circuit_gradient(params,sampled_pool)

for i in range(len(sampled_pool)):
    if abs(grads[i]) > 1.0e-10:
        print(f"Excitation : {sampled_pool[i]}, Gradient: {grads[i]}")